In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/drive/MyDrive/'Agricultural robot'/Coffee_kaggle_common.zip /content/

In [ ]:
!pip install ultralytics

In [ ]:
!unzip -q /content/Coffee_kaggle_common.zip -d /content/Coffee_kaggle_common

In [ ]:
from ultralytics import YOLO
m = YOLO('/content/drive/MyDrive/Agricultural robot/runs/croppie_baseline_v1/weights/best.pt')  # adjust path
r = m.val(data='/content/Coffee_kaggle_common/dataset.yaml', split='test',
          imgsz=640, batch=4, plots=True,
          project='/content/drive/MyDrive/runs', name='kaggle_crossdomain_v1')
print('mAP50', r.box.map50, 'mAP50-95', r.box.map)
for i, c in enumerate(r.box.ap_class_index):
    print(m.names[c], 'AP50', round(r.box.ap50[i], 3), 'AP50-95', round(r.box.ap[i], 3))

In [ ]:
!cd /content/ && zip -r /content/Coffee_kaggle_common.zip Coffee_kaggle_common

In [ ]:
!mv /content/Coffee_kaggle_common.zip /content/drive/MyDrive/'Agricultural robot'/Coffee_kaggle_common.zip

In [ ]:
   from ultralytics import YOLO
   for s in (320, 640, 960):
       m = YOLO('/content/drive/MyDrive/Agricultural robot/runs/croppie_baseline_v1/weights/best.pt')  # adjust path
       r = m.val(data='/content/Coffee_kaggle_common/dataset.yaml', split='test',
                 imgsz=s, batch=4, plots=False)
       print(s, round(r.box.map50, 3), [round(x, 3) for x in r.box.ap50])

In [ ]:
# Test 1: where do green's false positives go? (conf=0.25 only for this matrix; don't quote AP from this run)
import pandas as pd
r = m.val(data='/content/Coffee_kaggle_common/dataset.yaml', split='test',
          imgsz=640, batch=4, conf=0.25, plots=False)
lab = [m.names[i] for i in range(5)] + ['background']
print(pd.DataFrame(r.confusion_matrix.matrix.astype(int),
                   index=['pred '+l for l in lab], columns=['true '+l for l in lab]))

# Test 2: are Kaggle boxes a different size from Croppie's?
import glob, os, numpy as np
from PIL import Image
short = []
for f in glob.glob('/content/Coffee_kaggle_common/labels/*.txt'):
    stem = os.path.splitext(os.path.basename(f))[0]
    W, H = Image.open(glob.glob(f'/content/Coffee_kaggle_common/images/{stem}.*')[0]).size
    s = 640 / max(W, H)
    for l in open(f):
        p = l.split()
        if len(p) == 5:
            short.append(min(float(p[3]) * W * s, float(p[4]) * H * s))
short = np.array(short)
print(len(short), 'boxes | median short side at 640:', round(np.median(short), 1),
      'px | share under 16 px:', round((short < 16).mean(), 3))

In [12]:

import glob, os
from collections import Counter
from PIL import Image, ImageDraw, ImageOps

D = '/content/Coffee_kaggle_common'

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / (ua + 1e-9)

npred, fps = Counter(), []
for res in m.predict(source=f'{D}/images', imgsz=640, conf=0.5, stream=True, verbose=False):
    stem = os.path.splitext(os.path.basename(res.path))[0]
    gt = []
    for line in open(f'{D}/labels/{stem}.txt'):
        p = line.split()
        if len(p) == 5:
            c = int(p[0]); x, y, w, h = map(float, p[1:])
            gt.append((c, (x-w/2, y-h/2, x+w/2, y+h/2)))
    for b, c, s in zip(res.boxes.xyxyn.cpu().numpy(),
                       res.boxes.cls.cpu().numpy().astype(int),
                       res.boxes.conf.cpu().numpy()):
        npred[c] += 1
        if any(g[0] == c and iou(b, g[1]) >= 0.5 for g in gt):
            continue                      # correct detection
        other = [g[0] for g in gt if iou(b, g[1]) >= 0.5]
        fps.append((float(s), int(c), res.path, b, other[0] if other else -1, gt))

print('predictions at conf>=0.5:', {m.names[c]: n for c, n in sorted(npred.items())})
for c in sorted(npred):
    mine = [f for f in fps if f[1] == c]
    kinds = Counter('no label there' if f[4] == -1 else 'labelled ' + m.names[f[4]] for f in mine)
    print(f'{m.names[c]}: false positives {len(mine)}', dict(kinds))

out = '/content/drive/MyDrive/runs/kaggle_gallery'
os.makedirs(out, exist_ok=True)
COL = {1: (0, 200, 0), 3: (255, 0, 0), 4: (255, 215, 0)}
for c in (1, 3, 4):
    for s, _, path, b, other, gt in sorted([f for f in fps if f[1] == c], key=lambda f: -f[0])[:7]:
        im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
        im.thumbnail((1000, 1000)); w, h = im.size; d = ImageDraw.Draw(im)
        for g in gt:
            d.rectangle([g[1][0]*w, g[1][1]*h, g[1][2]*w, g[1][3]*h], outline=(255, 255, 255), width=2)
        d.rectangle([b[0]*w, b[1]*h, b[2]*w, b[3]*h], outline=COL[c], width=5)
        im.save(f'{out}/{m.names[c]}_{s:.2f}_{os.path.basename(path)}', quality=85)

predictions at conf>=0.5: {'dark_brown_cherry': 7, 'green_cherry': 8331, 'red_cherry': 1134, 'yellow_cherry': 270}
dark_brown_cherry: false positives 7 {'no label there': 7}
green_cherry: false positives 7503 {'no label there': 7429, 'labelled yellow_cherry': 72, 'labelled red_cherry': 2}
red_cherry: false positives 734 {'labelled yellow_cherry': 142, 'no label there': 592}
yellow_cherry: false positives 112 {'no label there': 109, 'labelled green_cherry': 1, 'labelled red_cherry': 2}


In [13]:

import random
random.seed(0)
pool = [f for f in fps if f[1] == 1 and f[4] == -1]      # green false positives, "no label there"
rout = '/content/drive/MyDrive/runs/kaggle_gallery_random'
os.makedirs(rout, exist_ok=True)
for i, (s, _, path, b, other, gt) in enumerate(random.sample(pool, 20)):
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    im.thumbnail((1000, 1000)); w, h = im.size; d = ImageDraw.Draw(im)
    for g in gt:
        d.rectangle([g[1][0]*w, g[1][1]*h, g[1][2]*w, g[1][3]*h], outline=(255, 255, 255), width=2)
    d.rectangle([b[0]*w, b[1]*h, b[2]*w, b[3]*h], outline=(0, 255, 0), width=5)
    im.save(f'{rout}/{i:02d}_{s:.2f}_{os.path.basename(path)}', quality=85)
print(len(pool), 'candidates; wrote 20 random ones')

7429 candidates; wrote 20 random ones


In [9]:
!rm -rf /content/Coffee_kaggle_common
!unzip -q /content/Coffee_kaggle_common.zip -d /content
!ls /content/Coffee_kaggle_common
!ls /content/Coffee_kaggle_common/images | wc -l

check_samples  conversion_log.txt  dataset.yaml  images  labels
432


In [11]:
!pip install -q ultralytics
from ultralytics import YOLO
m = YOLO('/content/drive/MyDrive/Agricultural robot/runs/croppie_baseline_v1/weights/best.pt')
print(m.names)

{0: 'dark_brown_cherry', 1: 'green_cherry', 2: 'low_visibility_unsure', 3: 'red_cherry', 4: 'yellow_cherry'}


In [14]:
import glob, numpy as np

def oriented_size(p):
    im = Image.open(p); W, H = im.size
    if im.getexif().get(274, 1) in (5, 6, 7, 8):   # phone rotation flag
        W, H = H, W
    return W, H

cache = {}
def short_px(path, w, h):        # w, h are normalised box width and height
    if path not in cache: cache[path] = oriented_size(path)
    W, H = cache[path]; s = 640 / max(W, H)
    return min(w * W * s, h * H * s)

fp = [short_px(f[2], f[3][2]-f[3][0], f[3][3]-f[3][1]) for f in pool]
gtsz = []
for lf in glob.glob(D + '/labels/*.txt'):
    stem = os.path.splitext(os.path.basename(lf))[0]
    ip = glob.glob(f'{D}/images/{stem}.*')[0]
    for l in open(lf):
        p = l.split()
        if len(p) == 5 and p[0] == '1':      # class 1 = green
            gtsz.append(short_px(ip, float(p[3]), float(p[4])))
print('unlabelled green detections: median', round(np.median(fp), 1), 'px, n =', len(fp))
print('labelled green:              median', round(np.median(gtsz), 1), 'px, n =', len(gtsz))

unlabelled green detections: median 25.3 px, n = 7429
labelled green:              median 44.0 px, n = 1199


In [15]:
def inter(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return iw * ih
def area(a): return (a[2]-a[0]) * (a[3]-a[1])

best_iou, inside, ratio, same_cls = [], [], [], []
for s, _, path, b, other, gt in pool:
    top = max(((inter(b, g[1]), g) for g in gt), key=lambda t: t[0], default=(0, None))
    i, g = top
    if g is None:
        best_iou.append(0); inside.append(0); continue
    best_iou.append(i / (area(b) + area(g[1]) - i + 1e-9))
    inside.append(i / (area(b) + 1e-9))          # share of the detection lying inside the label
    ratio.append(area(b) / area(g[1]))           # detection area / label area
    same_cls.append(g[0] == 1)

import numpy as np
bi, ins = np.array(best_iou), np.array(inside)
print('n', len(bi))
print('best IoU with any label >= 0.1:', round((bi >= 0.1).mean(), 3),
      '| >= 0.3:', round((bi >= 0.3).mean(), 3), '| < 0.1 (truly alone):', round((bi < 0.1).mean(), 3))
print('detection mostly inside a label (>=80% of its area):', round((ins >= 0.8).mean(), 3))
print('median detection/label area ratio:', round(float(np.median(ratio)), 2))
print('nearest label is green:', round(float(np.mean(same_cls)), 3))

n 7429
best IoU with any label >= 0.1: 0.27 | >= 0.3: 0.036 | < 0.1 (truly alone): 0.73
detection mostly inside a label (>=80% of its area): 0.177
median detection/label area ratio: 0.2
nearest label is green: 0.711
